# Бізнес-контекст
Компанія HealthRisk Analytics будує прототип моделі, яка за простими показниками (наприклад, вік та індекс маси тіла) прогнозує рівень страхового ризику клієнта.
Ваше завдання — змоделювати основу цієї системи, використовуючи PyTorch: спочатку — один лінійний нейрон, потім — малу нейромережу.

# Task 1
Створіть тензори та виконайте базові операції.
* Уявіть, що кожен тензор — це стовпець даних про клієнтів (вік, вага, зріст).
* Створіть 1D, 2D і 3D тензори(torch.manual_seed(42)).
* Виконайте операції додавання і поелементного множення, щоб обчислити умовний показник ризику.
* Увімкніть requires_grad=True для одного тензора, перевірте наявність grad_

In [1]:
import torch

In [2]:
torch.manual_seed(42)
age = torch.randint(18, 70, (5,), dtype=torch.float32)

In [3]:
age

tensor([36., 65., 50., 48., 48.])

Tensor it is fundomental option of matrix, with multidimencions

In [4]:
age = torch.randn(5)
print("Start values (normal distributed): ", age)
age = 44 + 26.0 * age
print("After processing: ", age) 

Start values (normal distributed):  tensor([ 0.3258, -0.8676,  1.5231,  0.6647, -1.0324])
After processing:  tensor([52.4695, 21.4417, 83.6015, 61.2813, 17.1564])


In [5]:
age_weight_height = torch.randn(5,3, requires_grad=True)
age_weight_height

tensor([[-0.2770, -0.1671, -0.1079],
        [-1.4285, -0.2810,  0.7489],
        [ 1.1164,  1.2931,  0.4137],
        [-0.5710, -0.9749,  0.1863],
        [ 1.6273,  1.1214, -0.6605]], requires_grad=True)

In [6]:
with torch.no_grad():
    age_weight_height[:,0] = 44.0 + 26.0 * age_weight_height[:,0]
    age_weight_height[:,1] = 95.0 + 25.0 * age_weight_height[:,1]
    age_weight_height[:,2] = 180.0 + 15.0 * age_weight_height[:,2]
age_weight_height

tensor([[ 36.7971,  90.8216, 178.3816],
        [  6.8581,  87.9757, 191.2339],
        [ 73.0270, 127.3272, 186.2059],
        [ 29.1540,  70.6263, 182.7952],
        [ 86.3088, 123.0361, 170.0919]], requires_grad=True)

In [7]:
risk = age_weight_height[:,0] * age_weight_height[:,1] + 0.5 * age_weight_height[:,2]
risk = (risk - risk.min()) / (risk.max() - risk.min())
risk

tensor([0.2731, 0.0000, 0.8688, 0.1451, 1.0000], grad_fn=<DivBackward0>)

# Task 2
Реалізуйте персептрон для передбачення ризику клієнта.
* Згенеруйте дані (torch.randn, 100 зразків, 1 ознака — індекс маси тіла).
* Цільова змінна — умовний "ризик" (лінійна комбінація + шум).
* Побудуйте модель nn.Linear(1, 1) і функцію втрат MSE.
* Виконайте 20 епох навчання з ручним оновленням параметрів.
* Виведіть початкові та фінальні значення ваг і loss.

In [8]:
bmi = torch.randn(100, 1)
print(bmi[:5])

tensor([[ 0.6872],
        [-1.0892],
        [-0.3553],
        [-0.9138],
        [-0.6581]])


In [9]:
noise = 0.1 * torch.randn(100, 1)
risk = 0.1 * bmi + noise
print(risk[:5])

tensor([[ 0.0916],
        [-0.1064],
        [-0.0701],
        [-0.0627],
        [-0.1389]])


In [ ]:
import torch.nn as nn
import torch.optim as optim

class Perceptron(nn.Module) : 
    def __init__(self):
        super(Perceptron, self).__init__()
        self.layer1 = nn.Linear(1,1)
        self.activator = nn.Sigmoid()

    def forward(self, x) :
        x = self.layer1(x)
        x = self.activator(x)
        return x

loss_f = nn.functional.mse_loss

In [15]:
model = Perceptron()
learning_rate = 0.1
output = model(bmi)
loss = loss_f(output, risk)
for param in model.parameters():
    print(f"Start epoch: {param}")
print(f'loss: {loss.item():.4f}')

print("\n")

for epoch in range(20) :
    output = model(bmi)
    loss = loss_f(output, risk)
    loss.backward()

    with torch.no_grad() :
        for param in model.parameters() :
            param -= learning_rate * param.grad
            param.grad.zero_()
    print(f"Epoch: {epoch}, loss: {loss.item():.4f}")

Start epoch: Parameter containing:
tensor([[-0.5995]], requires_grad=True)
Start epoch: Parameter containing:
tensor([-0.0875], requires_grad=True)
loss: 0.2689


Epoch: 0, loss: 0.2689
Epoch: 1, loss: 0.2638
Epoch: 2, loss: 0.2589
Epoch: 3, loss: 0.2540
Epoch: 4, loss: 0.2491
Epoch: 5, loss: 0.2443
Epoch: 6, loss: 0.2396
Epoch: 7, loss: 0.2350
Epoch: 8, loss: 0.2304
Epoch: 9, loss: 0.2259
Epoch: 10, loss: 0.2215
Epoch: 11, loss: 0.2171
Epoch: 12, loss: 0.2128
Epoch: 13, loss: 0.2086
Epoch: 14, loss: 0.2045
Epoch: 15, loss: 0.2004
Epoch: 16, loss: 0.1964
Epoch: 17, loss: 0.1925
Epoch: 18, loss: 0.1887
Epoch: 19, loss: 0.1850


In [16]:
for param in model.parameters():
    print(f"End epoch: {param}")
print(f'loss: {loss.item():.4f}')

End epoch: Parameter containing:
tensor([[-0.4464]], requires_grad=True)
End epoch: Parameter containing:
tensor([-0.4772], requires_grad=True)
loss: 0.1850
